In [1]:
from websocket import create_connection
import json
import numpy as np
import pandas as pd
from datetime import date
from time import sleep
from tqdm import tqdm

In [34]:
## web socket url
socket = 'wss://data.tradingview.com/socket.io/websocket'
# change session id based on current session value
session_id = 'cs_hXWsYYp3aGH1'
## example messages to be sent
# ~m~55~m~{"m":"chart_create_session","p":["cs_yDp6P0icM81I",""]}
# ~m~117~m~{"m":"resolve_symbol","p":["cs_yDp6P0icM81I","sds_sym_1","={\"adjustment\":\"splits\",\"symbol\":\"NSE:HDFCBANK\"}"]}
# ~m~82~m~{"m":"create_series","p":["cs_yDp6P0icM81I","sds_1","s1","sds_sym_1","1W",300,""]}

downloads = './data/nifty500_5yr_daily/'

# assuming 5 days every week from 2020 Jan 1st
count_days = int(np.ceil(len(pd.date_range(start='2021-10-01', end=date.today())) * 5/7))

In [35]:
# create message as per template and send
def create_msg(ws, fun, arg):
    ms = json.dumps({"m":fun,"p":arg})
    msg = f'~m~{len(ms)}~m~{ms}'
    ws.send(msg)

# format result data into pandas dataframe
def format_data(data):
    start = data.find('"s":[')
    end = data.find(',"ns":')
    trimmed = data[start+4 : end]
    final_data = [x["v"] for x in json.loads(data[start+4 : end])]
    # print(final_data)
    return final_data

In [36]:
def fetch_historic_data(symbol):
    # resolve_symbol argument
    sess_msg = [session_id,""]    
    resolve_symbol = '={\"adjustment\":\"splits\",\"symbol\":\"NSE:' + symbol + '\"}'
    res_sym_msg = [session_id,"sds_sym_1",resolve_symbol]
    ser_msg = [session_id,"sds_1","s1","sds_sym_1","1D",count_days,""]
    # create fresh websocket connection
    ws = create_connection(socket)
    # send messages to websocket
    create_msg(ws, 'chart_create_session', sess_msg)
    create_msg(ws, 'resolve_symbol', res_sym_msg)
    create_msg(ws, 'create_series', ser_msg)
    
    while True :
        # receive response for the messages sent
        res = ws.recv()
        if "series_completed" in res:
            formatted_data = format_data(res)
            # convert json to pandas dataframe for analysis
            df_result = pd.DataFrame(formatted_data, columns=['date', 'open', 'high', 'low', 'close', 'volume'])
            # convert epoch timestamp to date yyyy-mm-dd format
            df_result['date'] = df_result['date'].apply(lambda x : str(date.fromtimestamp(x)))
            df_result.sort_values('date', inplace=True)
            # calculate average
            df_result['avg'] = df_result.apply(lambda row : int((row['open'] + row['high'] + row['low'] + row['close'])/4), axis=1)
            # shutdown websocket connection for next iteration to work properly
            ws.shutdown()
            # break the infinite while loop
            break
    
    return df_result

In [13]:
# run for a sample symbol
df_result = fetch_historic_data('SBIN')
df_result

,date,open,high,low,close,volume,avg
0,2021-07-16,432.00000,432.50000,427.45001,430.00000,8713863.0,430
1,2021-07-19,423.39999,429.45001,418.85001,427.89999,14426686.0,424
2,2021-07-20,427.00000,427.00000,418.89999,420.89999,14623321.0,423
3,2021-07-22,425.54999,426.70001,420.89999,422.04999,11799761.0,423
4,2021-07-23,422.89999,429.95001,419.50000,428.89999,17704661.0,425
...,...,...,...,...,...,...,...
1051,2025-10-10,862.10000,883.75000,861.30000,880.65000,13713015.0,871
1052,2025-10-13,879.00000,888.10000,876.00000,882.95000,11923143.0,881
1053,2025-10-14,883.00000,884.15000,872.10000,876.95000,7681973.0,879
1054,2025-10-15,878.10000,887.50000,877.05000,886.10000,8269225.0,882


In [25]:
# use index/list of your choice
#/Users/abss/Desktop/Work/StockPrediction/share_analysis
# using shortlist generated from clustering.ipynb
shortlist = pd.read_csv('./data/shortlist.csv')
shortlist['Ticker'] = shortlist['Ticker'].str.replace('-', '_')
# stocks which have number ticker are not working, so exclude them
shortlist = shortlist.loc[~shortlist['Ticker'].str.contains(r'[0-9]{5}')]
shortlist

,Short_Name,Ticker
0,St Bk of India,SBIN
1,HDFC Bank,HDFCBANK
2,Reliance Industr,RELIANCE
3,ICICI Bank,ICICIBANK
4,TCS,TCS
...,...,...
371,SignatureGlobal,SIGNATURE
372,Sobha,SOBHA
373,FSN E-Commerce,NYKAA
374,PTC Industries,PTCIL


In [ ]:
# if loop breaks before completing all symbols, add the completed symbols to this list
completed_list = []

In [50]:
# these stocks are not working for unknown reason, excluding them too
completed_list.extend(['BIRET', 'EMBASSY', 'ISEC', 'IWEL', 'MINDSPACE', 'NXST', 'ZOMATO', 'VTL'])
symbols = list(sorted(set(shortlist['Ticker']) - set(completed_list)))

for symbol in (pbar:= tqdm(symbols)):
    pbar.set_postfix_str(symbol)
    if '-' in symbol:
        symbol = symbol.replace('-', '_')
    try:
        df_result = fetch_historic_data(symbol)
        df_result.to_csv(downloads + symbol + '.csv', index=None)
        completed_list.append(symbol)
        sleep(1)
    except:
        print(symbol)
        raise

 50%|███████████████████▌                   | 1/2 [00:13<00:13, 13.81s/it,  VTL]

 VTL


KeyboardInterrupt: 

In [41]:
symbols

['EMBASSY',
 'EMCURE',
 'ENDURANCE',
 'ERIS',
 'ESCORTS',
 'EXIDEIND',
 'FEDERALBNK',
 'FINCABLES',
 'FINEORG',
 'FIVESTAR',
 'FLUOROCHEM',
 'FORTIS',
 'FSL',
 'GAIL',
 'GESHIP',
 'GICRE',
 'GILLETTE',
 'GLAND',
 'GLAXO',
 'GODFRYPHLP',
 'GODIGIT',
 'GODREJPROP',
 'GPIL',
 'GRASIM',
 'GRAVITA',
 'GRINDWELL',
 'GRSE',
 'GSPL',
 'GUJGASLTD',
 'GVT&D',
 'HAL',
 'HAPPSTMNDS',
 'HATSUN',
 'HAVELLS',
 'HBLENGINE',
 'HCLTECH',
 'HDFCAMC',
 'HDFCBANK',
 'HDFCLIFE',
 'HEROMOTOCO',
 'HFCL',
 'HINDALCO',
 'HINDCOPPER',
 'HINDPETRO',
 'HINDUNILVR',
 'HINDZINC',
 'HONAUT',
 'HSCL',
 'HUDCO',
 'ICICIBANK',
 'ICICIGI',
 'IDBI',
 'IDFCFIRSTB',
 'IEX',
 'IGL',
 'IIFL',
 'INDGN',
 'INDHOTEL',
 'INDIAMART',
 'INDIANB',
 'INDIGO',
 'INDIGRID',
 'INDUSINDBK',
 'INDUSTOWER',
 'INFY',
 'INGERRAND',
 'INOXWIND',
 'IOB',
 'IPCALAB',
 'IRB',
 'IRCTC',
 'IREDA',
 'IRFC',
 'ISEC',
 'ITC',
 'IWEL',
 'J&KBANK',
 'JAIBALAJI',
 'JBCHEPHARM',
 'JBMA',
 'JINDALSAW',
 'JIOFIN',
 'JKCEMENT',
 'JSWHL',
 'JSWINFRA',
 'JUBL

In [52]:
len(completed_list), completed_list

(431,
 ['360ONE',
  '3MINDIA',
  'AADHARHFC',
  'AARTIIND',
  'AAVAS',
  'ABB',
  'ABBOTINDIA',
  'ABCAPITAL',
  'ABSLAMC',
  'ACC',
  'ACE',
  'ADANIENT',
  'ADANIGREEN',
  'ADANIPORTS',
  'ADANIPOWER',
  'AETHER',
  'AFFLE',
  'AIIL',
  'AJANTPHARM',
  'AKZOINDIA',
  'ALKEM',
  'AMBER',
  'AMBUJACEM',
  'ANANDRATHI',
  'ANANTRAJ',
  'ANGELONE',
  'APARINDS',
  'APLAPOLLO',
  'APLLTD',
  'APOLLOHOSP',
  'APOLLOTYRE',
  'APTUS',
  'ARE&M',
  'ASAHIINDIA',
  'ASHOKLEY',
  'ASTRAL',
  'ATGL',
  'ATUL',
  'AUBANK',
  'AUROPHARMA',
  'AVANTIFEED',
  'AWL',
  'AXISBANK',
  'BAJAJ_AUTO',
  'BAJAJFINSV',
  'BAJAJHFL',
  'BAJAJHLDNG',
  'BAJFINANCE',
  'BALKRISIND',
  'BANDHANBNK',
  'BANKBARODA',
  'BANKINDIA',
  'BASF',
  'BATAINDIA',
  'BBTC',
  'BDL',
  'BEL',
  'BERGEPAINT',
  'BHARATFORG',
  'BHARTIARTL',
  'BHARTIHEXA',
  'BIKAJI',
  'BIOCON',
  'BAJAJ_AUTO',
  'BAJAJ_AUTO',
  'BIRET',
  'BAJAJ_AUTO',
  'BLS',
  'BLUEDART',
  'BLUEJET',
  'BLUESTARCO',
  'BOSCHLTD',
  'BRIGADE',
  'BRIT